# Laborator 9 — Agent pentru îngrijirea unei grădini comunitare

**Disciplina:** Inteligență Artificială Generativă  
**Autor:** Carp Dan  
**Tema:** utilizarea instrumentelor Python și SQLite pentru observarea umidității solului și formularea unui plan de îngrijire.

> Datele despre plante și măsurători sunt fictive, create pentru această demonstrație. Lucrarea este o aplicație nouă, construită în jurul unei probleme diferite de exemplele consultate.

## Context

Într-o grădină comunitară, măsurătorile sunt utile numai dacă pot fi transformate în acțiuni ușor de verificat. Am construit un agent local care poate consulta plantele, înregistra o măsurătoare și alcătui o listă de plante care ar putea avea nevoie de apă. Agentul nu udă plantele și nu pretinde că pragurile din exemplu sunt recomandări horticole universale.

## Obiective

1. Organizarea datelor demonstrative într-o bază SQLite.
2. Construirea a trei instrumente cu responsabilități clare.
3. Implementarea unui ciclu transparent de decizie, apelare a instrumentului și răspuns.
4. Verificarea logicii cu teste reproductibile, inclusiv date invalide.
5. Analiza rezultatului și explicarea limitelor soluției.

**Notă despre agent:** pentru ca notebook-ul să funcționeze fără servicii externe, planificatorul din această versiune folosește reguli explicite pentru câteva formulări de test. Așadar, agentul demonstrează fluxul de lucru și instrumentele, nu selecția autonomă făcută de un LLM. Conectarea unui model real este propusă ca extensie, nu este prezentată ca o execuție efectuată.

## 1. Arhitectura și regulile de lucru

Fluxul este: cerere → planificator local → un instrument Python → SQLite → observație → răspuns. Instrumentele sunt singura cale prin care agentul accesează datele.

| Instrument | Responsabilitate |
|---|---|
| `inspect_plants` | Citește plantele, cu filtrare opțională după zonă. |
| `record_moisture` | Validează și salvează o măsurătoare; actualizează valoarea curentă. |
| `get_watering_plan` | Listează plantele sub pragul demonstrativ de umiditate. |

Pragurile sunt proprietăți ale datelor demonstrative, nu adevăruri agronomice generale. Planul este consultativ: nu execută udarea și nu înlocuiește evaluarea unei persoane.

## 2. Pregătirea mediului

Notebook-ul utilizează doar biblioteca standard Python (`sqlite3`, `datetime`, `re`, `unittest` și `json`). Nu este necesară o cheie API și nici instalarea unui pachet extern. Baza de date se recreează la inițializare pentru rezultate reproductibile.

In [ ]:
import json
import re
import sqlite3
import unittest
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

DB_PATH = Path("gradina_comunitara.db")
DATE_FORMAT = "%Y-%m-%d %H:%M"

print("Baza demonstrativă:", DB_PATH.resolve())

## 3. Crearea bazei de date

Tabelul `plants` păstrează starea curentă și pragul definit pentru fiecare plantă. Tabelul `moisture_readings` păstrează istoricul, astfel încât înregistrarea unei valori noi să nu șteargă măsurătorile anterioare. Constrângerile SQL completează validarea realizată în Python.

In [ ]:
SAMPLE_PLANTS = [
    (1, "Busuioc", "A", 30, 18),
    (2, "Roșii", "A", 35, 42),
    (3, "Mentă", "B", 32, 24),
    (4, "Lavandă", "B", 20, 15),
    (5, "Ardei", "A", 33, 28),
    (6, "Salată", "C", 28, 19),
    (7, "Rozmarin", "B", 25, 39),
    (8, "Spanac", "C", 30, 31),
]
INITIAL_TIME = "2026-11-08 09:00"

@contextmanager
def connection():
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    con.execute("PRAGMA foreign_keys = ON")
    try:
        yield con
        con.commit()
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()

def initialize_database():
    with connection() as con:
        con.executescript("""
            DROP TABLE IF EXISTS moisture_readings;
            DROP TABLE IF EXISTS plants;
            CREATE TABLE plants (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL UNIQUE,
                zone TEXT NOT NULL,
                min_moisture INTEGER NOT NULL CHECK(min_moisture BETWEEN 0 AND 100),
                moisture_pct INTEGER NOT NULL CHECK(moisture_pct BETWEEN 0 AND 100),
                last_checked TEXT NOT NULL
            );
            CREATE TABLE moisture_readings (
                id INTEGER PRIMARY KEY,
                plant_id INTEGER NOT NULL REFERENCES plants(id),
                measured_at TEXT NOT NULL,
                moisture_pct INTEGER NOT NULL CHECK(moisture_pct BETWEEN 0 AND 100)
            );
        """)
        con.executemany(
            """INSERT INTO plants(id, name, zone, min_moisture, moisture_pct, last_checked)
               VALUES (?, ?, ?, ?, ?, ?)""",
            [(*plant, INITIAL_TIME) for plant in SAMPLE_PLANTS],
        )
        con.executemany(
            """INSERT INTO moisture_readings(plant_id, measured_at, moisture_pct)
               VALUES (?, ?, ?)""",
            [(plant[0], INITIAL_TIME, plant[4]) for plant in SAMPLE_PLANTS],
        )

initialize_database()
with connection() as con:
    rows = [dict(row) for row in con.execute(
        "SELECT id, name, zone, min_moisture, moisture_pct FROM plants ORDER BY id"
    )]
print(f"Plante încărcate: {len(rows)}")
print(json.dumps(rows, ensure_ascii=False, indent=2))

## 4. Implementarea instrumentelor

Toate valorile venite din exterior sunt validate înainte de modificare. Interogările folosesc parametri SQL, nu concatenarea textului primit în instrucțiunea SQL. Rezultatele sunt dicționare simple, ușor de verificat și de transmis mai departe.

In [ ]:
TOOL_LOG = []

def inspect_plants(zone=""):
    zone = zone.strip().upper()
    query = "SELECT id, name, zone, min_moisture, moisture_pct, last_checked FROM plants"
    params = ()
    if zone:
        query += " WHERE zone = ?"
        params = (zone,)
    query += " ORDER BY id"
    with connection() as con:
        plants = [dict(row) for row in con.execute(query, params)]
    result = {"count": len(plants), "plants": plants}
    TOOL_LOG.append({"tool": "inspect_plants", "zone": zone, "count": len(plants)})
    return result

def record_moisture(plant_id, moisture_pct, measured_at):
    if isinstance(plant_id, bool) or not isinstance(plant_id, int):
        raise ValueError("ID-ul plantei trebuie să fie un număr întreg.")
    if isinstance(moisture_pct, bool) or not isinstance(moisture_pct, int):
        raise ValueError("Umiditatea trebuie să fie un număr întreg între 0 și 100.")
    if not 0 <= moisture_pct <= 100:
        raise ValueError("Umiditatea trebuie să fie între 0 și 100.")
    try:
        parsed_time = datetime.strptime(measured_at, DATE_FORMAT)
    except (TypeError, ValueError) as exc:
        raise ValueError("Data trebuie să respecte formatul YYYY-MM-DD HH:MM.") from exc
    normalized_time = parsed_time.strftime(DATE_FORMAT)

    with connection() as con:
        plant = con.execute("SELECT id, name FROM plants WHERE id = ?", (plant_id,)).fetchone()
        if plant is None:
            raise ValueError(f"Nu există o plantă cu ID-ul {plant_id}.")
        con.execute(
            "INSERT INTO moisture_readings(plant_id, measured_at, moisture_pct) VALUES (?, ?, ?)",
            (plant_id, normalized_time, moisture_pct),
        )
        con.execute(
            "UPDATE plants SET moisture_pct = ?, last_checked = ? WHERE id = ?",
            (moisture_pct, normalized_time, plant_id),
        )
    result = {
        "success": True, "plant_id": plant_id, "name": plant["name"],
        "moisture_pct": moisture_pct, "measured_at": normalized_time,
    }
    TOOL_LOG.append({"tool": "record_moisture", "plant_id": plant_id, "moisture_pct": moisture_pct})
    return result

def get_watering_plan():
    with connection() as con:
        plants = [dict(row) for row in con.execute("""
            SELECT id, name, zone, min_moisture, moisture_pct, last_checked
            FROM plants
            WHERE moisture_pct < min_moisture
            ORDER BY (min_moisture - moisture_pct) DESC, id
        """)]
    recommendations = [
        {
            **plant,
            "deficit_pct": plant["min_moisture"] - plant["moisture_pct"],
            "recommendation": "Verifică solul și evaluează dacă este necesară udarea.",
        }
        for plant in plants
    ]
    result = {"count": len(recommendations), "plants": recommendations}
    TOOL_LOG.append({"tool": "get_watering_plan", "count": len(recommendations)})
    return result

TOOLS = {
    "inspect_plants": inspect_plants,
    "record_moisture": record_moisture,
    "get_watering_plan": get_watering_plan,
}
print("Instrumente disponibile:", ", ".join(TOOLS))

## 5. Agentul local: decizie, acțiune, observație

Planificatorul de mai jos recunoaște câteva tipare românești definite explicit. El nu este un model lingvistic: dacă formularea nu se potrivește, întoarce o eroare clară în loc să ghicească. În cererea de înregistrare, ID-ul și umiditatea sunt extrase dintr-un format intenționat restrâns.

Acest compromis face laboratorul reproductibil și permite observarea apelurilor reale ale instrumentelor. Pentru un agent LLM, planificatorul ar putea fi înlocuit cu function calling, păstrând aceleași instrumente validate.

In [ ]:
def run_agent(request):
    text = request.strip()
    normalized = text.casefold()

    if (
        "planul" in normalized
        or "plan de" in normalized
        or "udat" in normalized
        or "nevoie de apă" in normalized
        or "irigare" in normalized
    ):
        result = get_watering_plan()
        names = ", ".join(plant["name"] for plant in result["plants"]) or "Nicio plantă"
        answer = f"Am găsit {result['count']} plante sub pragul demonstrativ: {names}."
        tool_name = "get_watering_plan"
    elif "înregistrează" in normalized or "măsurătoare" in normalized:
        match = re.search(
            r"planta\s+(\d+)\D+?(\d{1,3})\s*%",
            text,
            flags=re.IGNORECASE,
        )
        if not match:
            raise ValueError(
                "Folosește formatul: Înregistrează planta 3 cu umiditatea 21%."
            )
        result = record_moisture(
            int(match.group(1)),
            int(match.group(2)),
            "2026-11-09 09:00",
        )
        answer = (
            f"Măsurătoarea pentru {result['name']} a fost salvată: "
            f"{result['moisture_pct']}%."
        )
        tool_name = "record_moisture"
    elif "plante" in normalized or "zona" in normalized:
        zone_match = re.search(r"zona\s+([a-z])\b", normalized, flags=re.IGNORECASE)
        result = inspect_plants(zone_match.group(1) if zone_match else "")
        answer = f"Am găsit {result['count']} plante în catalog."
        tool_name = "inspect_plants"
    else:
        raise ValueError("Nu am recunoscut solicitarea. Reformulează folosind plante, măsurătoare sau plan.")

    return {"request": text, "tool": tool_name, "observation": result, "answer": answer}

requests = [
    "Arată plantele din zona B.",
    "Înregistrează planta 3 cu umiditatea 21%.",
    "Pregătește planul plantelor care au nevoie de apă.",
]

for request in requests:
    trace = run_agent(request)
    print("Cerere:", trace["request"])
    print("Instrument:", trace["tool"])
    print("Observație:", trace["observation"])
    print("Răspuns:", trace["answer"], "\n")

## 6. Verificări automate

Testele verifică atât rezultatele așteptate, cât și comportamentul la intrări invalide. Înaintea fiecărui test recreez baza, astfel încât rezultatul să nu depindă de ordinea sau de rulările anterioare.

In [ ]:
class TestGardenAgent(unittest.TestCase):
    def setUp(self):
        initialize_database()
        TOOL_LOG.clear()

    def test_zone_filter(self):
        result = inspect_plants("a")
        self.assertEqual(result["count"], 3)
        self.assertTrue(all(plant["zone"] == "A" for plant in result["plants"]))

    def test_plan_uses_threshold(self):
        result = get_watering_plan()
        self.assertEqual(
            {plant["name"] for plant in result["plants"]},
            {"Busuioc", "Mentă", "Lavandă", "Ardei", "Salată"},
        )

    def test_record_updates_current_value_and_history(self):
        result = record_moisture(3, 40, "2026-11-09 10:15")
        self.assertEqual(result["moisture_pct"], 40)
        with connection() as con:
            current = con.execute(
                "SELECT moisture_pct FROM plants WHERE id = 3"
            ).fetchone()[0]
            history_count = con.execute(
                "SELECT COUNT(*) FROM moisture_readings WHERE plant_id = 3"
            ).fetchone()[0]
        self.assertEqual(current, 40)
        self.assertEqual(history_count, 2)

    def test_invalid_moisture_is_rejected(self):
        with self.assertRaises(ValueError):
            record_moisture(1, 101, "2026-11-09 10:15")

    def test_unknown_plant_is_rejected(self):
        with self.assertRaises(ValueError):
            record_moisture(999, 25, "2026-11-09 10:15")

    def test_agent_selects_expected_tool(self):
        self.assertEqual(run_agent("Arată plantele din zona C.")["tool"], "inspect_plants")
        self.assertEqual(
            run_agent("Înregistrează planta 1 cu umiditatea 20%.")["tool"],
            "record_moisture",
        )
        self.assertEqual(
            run_agent("Arată planul de udat.")["tool"],
            "get_watering_plan",
        )

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestGardenAgent)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Cel puțin un test a eșuat."
print(f"Rezultat: {test_result.testsRun} teste trecute.")

## 7. Analiza rezultatelor și limite

Raportul următor este calculat din baza de date, nu introdus manual. El arată numărul de plante pe zonă și numărul de recomandări produse de regulă curentă.

In [ ]:
with connection() as con:
    zone_counts = [
        dict(row) for row in con.execute(
            "SELECT zone, COUNT(*) AS plants FROM plants GROUP BY zone ORDER BY zone"
        )
    ]
plan = get_watering_plan()

print("Plante pe zonă:")
for row in zone_counts:
    print(f"  Zona {row['zone']}: {row['plants']}")
print(f"Plante sub pragul demonstrativ: {plan['count']} din {len(SAMPLE_PLANTS)}")
print("Apeluri înregistrate:", json.dumps(TOOL_LOG, ensure_ascii=False, indent=2))

## 8. Interpretare și concluzii

- Separarea accesului la date în instrumente mici a permis testarea funcțiilor fără un model extern.
- Istoricul măsurătorilor păstrează evoluția datelor, iar tabela principală oferă rapid valoarea curentă.
- Interogările parametrizate și limitele de domeniu reduc riscul unor valori invalide sau al unor comenzi SQL introduse prin text.
- Planul identifică numai valorile sub pragurile ilustrative; nu ia în calcul tipul solului, vremea, specia în detaliu sau alte condiții horticole.
- Routerul demonstrativ este bazat pe reguli și nu înțelege formulări arbitrare. Într-un produs, aș conecta un LLM prin function calling, aș limita instrumentele la operații permise și aș păstra validarea în Python.

**Concluzie personală:** laboratorul arată că partea importantă a unui agent nu este doar formularea răspunsului, ci și delimitarea acțiunilor pe care le poate efectua și verificarea rezultatului produs de fiecare instrument.

### Extensie propusă

Înlocuiește `run_agent` cu un model care alege dintre cele trei instrumente. Compară selecția modelului cu routerul local pentru formulări noi, păstrează jurnalul apelurilor și nu permite modelului să ocolească validările instrumentelor.